# Simulación interactiva de tenis de mesa

Ajusta los controles para recalcular la trayectoria y sus variables de estado. Las figuras se crean por separado para mantener una lectura clara.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Permite ejecutar el notebook desde la raíz del repositorio o desde notebooks/.
repository_root = Path.cwd()
if not (repository_root / 'src').is_dir():
    repository_root = repository_root.parent
sys.path.insert(0, str(repository_root / 'src'))

from table_tennis_sim import (
    InitialState,
    SimulationParameters,
    plot_all,
    simulate,
)

In [2]:
style = {'description_width': 'initial'}
layout = widgets.Layout(width='360px')

def slider(description, value, minimum, maximum, step):
    return widgets.FloatSlider(
        description=description, value=value, min=minimum, max=maximum, step=step,
        continuous_update=False, style=style, layout=layout, readout_format='.3g'
    )

vx = slider('Velocidad inicial x (mm/s)', 2500, -5000, 5000, 100)
vy = slider('Velocidad inicial y (mm/s)', 0, -3000, 3000, 100)
vz = slider('Velocidad inicial z (mm/s)', 1500, -4000, 4000, 100)
wx = slider('Velocidad angular x (rad/s)', 0, -500, 500, 10)
wy = slider('Velocidad angular y (rad/s)', 0, -500, 500, 10)
wz = slider('Velocidad angular z (rad/s)', 100, -500, 500, 10)
drag = slider('Arrastre', 2.7, 0, 10, 0.1)
magnus = slider('Efecto Magnus', 0.01, 0, 0.1, 0.001)
table_restitution = slider('Restitución de mesa', 0.77, 0, 1, 0.01)
net_restitution = slider('Restitución de red', 0.5, 0, 1, 0.01)
table_friction = slider('Fricción de mesa', 0.25, 0, 1, 0.01)

controls = {
    'vx': vx, 'vy': vy, 'vz': vz, 'wx': wx, 'wy': wy, 'wz': wz,
    'drag_value': drag, 'magnus_value': magnus,
    'table_restitution_value': table_restitution,
    'net_restitution_value': net_restitution,
    'table_friction_value': table_friction,
}

display(widgets.VBox([
    widgets.HTML('<h3>Estado inicial</h3>'), vx, vy, vz, wx, wy, wz,
    widgets.HTML('<h3>Parámetros físicos</h3>'),
    drag, magnus, table_restitution, net_restitution, table_friction,
]))

In [3]:
output = widgets.Output()

def regenerate_plots(vx, vy, vz, wx, wy, wz, drag_value, magnus_value,
                     table_restitution_value, net_restitution_value,
                     table_friction_value):
    parameters = SimulationParameters(
        drag=drag_value,
        magnus=magnus_value,
        table_restitution=table_restitution_value,
        net_restitution=net_restitution_value,
        table_friction=table_friction_value,
    )
    initial_state = InitialState(
        position=(600.0, 762.0, 1100.0),
        velocity=(vx, vy, vz),
        angular_velocity=(wx, wy, wz),
    )

    with output:
        output.clear_output(wait=True)
        figures = plot_all(simulate(parameters, initial_state))
        for figure in figures:
            display(figure)
        plt.close('all')

# La referencia mantiene activos los observadores de los controles.
interactive_plots = widgets.interactive_output(regenerate_plots, controls)
display(output)

Output()